In [4]:

import pandas as pd
import numpy as np
print("Capstone Project Started")


Capstone Project Started


In [5]:
project_df = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005","A002","A005","A003","A001","A003","A001","A004","A004","A005"],
    "Project": [f"Project {i}" for i in range(1,15)],
    "Cost": [1002000,2000000,4500000,5500000,np.nan,680000,400000,350000,np.nan,300000,2000000,1000000,3000000,2000000],
    "Status": ["Finished","Ongoing","Finished","Ongoing","Finished","Failed","Finished","Failed","Ongoing","Finished","Failed","Ongoing","Finished","Finished"]
})
project_df.to_csv("project.csv", index=False)

In [6]:
employee_df = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005"],
    "Name": ["John Alter","Alice Luxemburg","Tom Sabestine","Nina Adgra","Amy Johny"],
    "Gender": ["M","F","M","F","F"],
    "City": ["Paris","London","Berlin","Newyork","Madrid"],
    "Age": [25,27,29,31,30]
})
employee_df.to_csv("employee.csv", index=False)

In [7]:
seniority_df = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005"],
    "Designation Level": [2,2,3,2,3]
})
seniority_df.to_csv("seniority.csv", index=False)

In [8]:
costs = project_df["Cost"].tolist()
running_avg = 0
count = 0

for i in range(len(costs)):
    if pd.notna(costs[i]):
        running_avg = (running_avg * count + costs[i]) / (count + 1)
        count += 1
    else:
        costs[i] = running_avg

project_df["Cost"] = costs

In [9]:
employee_df[["First Name","Last Name"]] = employee_df["Name"].str.split(" ", expand=True)
employee_df.drop(columns="Name", inplace=True)

In [10]:
final_df = project_df.merge(employee_df, on="ID", how="left")
final_df = final_df.merge(seniority_df, on="ID", how="left")

In [11]:
final_df["Bonus"] = np.where(
    final_df["Status"] == "Finished",
    final_df["Cost"] * 0.05,
    0
)

In [12]:
final_df.loc[final_df["Status"] == "Failed", "Designation Level"] -= 1
final_df = final_df[final_df["Designation Level"] <= 4]

In [13]:
final_df["First Name"] = np.where(
    final_df["Gender"] == "M",
    "Mr. " + final_df["First Name"],
    "Mrs. " + final_df["First Name"]
)
final_df.drop(columns="Gender", inplace=True)

In [14]:
final_df.loc[final_df["Age"] > 29, "Designation Level"] += 1

In [15]:
total_cost_df = (
    final_df.groupby(["ID","First Name"])["Cost"]
    .sum()
    .reset_index(name="TotalProjCost")
)

In [16]:
result = final_df[final_df["City"].str.contains("o", case=False)]
print(result)

      ID     Project       Cost    Status     City  Age  First Name  \
1   A002   Project 2  2000000.0   Ongoing   London   27  Mrs. Alice   
3   A004   Project 4  5500000.0   Ongoing  Newyork   31   Mrs. Nina   
5   A002   Project 6   680000.0    Failed   London   27  Mrs. Alice   
11  A004  Project 12  1000000.0   Ongoing  Newyork   31   Mrs. Nina   
12  A004  Project 13  3000000.0  Finished  Newyork   31   Mrs. Nina   

    Last Name  Designation Level     Bonus  
1   Luxemburg                  2       0.0  
3       Adgra                  3       0.0  
5   Luxemburg                  1       0.0  
11      Adgra                  3       0.0  
12      Adgra                  3  150000.0  
